

**Objectif :** prédire l'influance sur la quantité de déchets verts
- Modèle : Linéraire Régression
- Jeu de données : `data_wip_v5.xlsx`

In [1]:
# 1. Import des librairies essentielles
import pandas as pd
import numpy as np
import plotly.express as px
import pickle
import json 
from pathlib import Path

# 2. Modèles et transformation de donnees
from sklearn.model_selection import train_test_split,GridSearchCV, cross_val_score,KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso,LassoCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error,mean_absolute_percentage_error
import statsmodels.api as sm
import warnings
from sklearn.exceptions import ConvergenceWarning

# 3. Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

import boto3, pickle
from io import BytesIO,StringIO

import datetime,os

from dotenv import load_dotenv
load_dotenv("../secrets.env")


True

# Fonctions et méthodes

In [2]:
def select_target(target, df):
    """
    Cette fonciton permet de retourner le X et Y en fonction des paramètres 
    target : Nom de la target
    df : Dataframe sur le lequel on doit separer la target (Y) des autres colonnes (X)
    """
    # On supprime l'ensemble de starget possible
    A_supp = ['Deblais_gravats','Dechets_verts','Encombrants','Materiaux_recyclables']

    Y = df[target]
    # X = df.drop(A_supp, axis=1)
    X = df.drop(target, axis=1)
    # Si des donnees ne sont pas presentes, on remplit avec la valeur mediane
    X= X.fillna(X.median(numeric_only=True))

    return X,Y

In [3]:
def Select_features_PValue(X_encoded,Y,target,new_rows):
    """
    Cette fonction permet de selectionner les features que l'on garde en fonction de la pvalue calculer via le model OLS
    Par convention, on supprime les features ayant une pvalue superieure à 0.05 (5%) une par une
    en partant de la pvalue la plusimportante à la moins importante
    Une fois cette selection effectuee, on calcule les R² score
    On lance une dernière fois le modèle afin que les paramètres soit sur l'ensemble des lignes.
    """
    X_encoded = X_encoded.copy()
    boucle = True
    while boucle:
        X_const=sm.add_constant(X_encoded)
        model = sm.OLS(Y,X_const)
        results = model.fit()

        results_df = pd.DataFrame({
            'coef': results.params,
            'std err': results.bse,
            't': results.tvalues,
            'P>|t|': results.pvalues
        })
        results_df_sorted = results_df[results_df['P>|t|'] > 0.05].sort_values(by="P>|t|", ascending=False)
        
        if len(results_df_sorted)>0 :
            feature_to_remove = results_df_sorted.index[0]
            if feature_to_remove != 'const':
                X_encoded = X_encoded.drop(feature_to_remove, axis=1)
            else:
                boucle = False
        else:
            boucle = False

    
    X_train,X_test,Y_train, Y_test = train_test_split(X_const,Y,random_state=random, test_size=size)
   
    model = sm.OLS(Y_train,X_train).fit()

    Y_test_pred = model.predict(X_test)
    Y_train_pred = model.predict(X_train)

    new_rows[0]['R2 train'] = r2_score(Y_train,Y_train_pred)
    new_rows[0]['R2 test'] = r2_score(Y_test,Y_test_pred)

    rmse = np.sqrt(mean_squared_error(Y_test,Y_test_pred))
    new_rows[0]['RMSE'] = round(np.sqrt(mean_squared_error(Y_test,Y_test_pred)),0)
    new_rows[0]['MAE'] = round(mean_absolute_error(Y_test,Y_test_pred),0)
    new_rows[0]['MAPE'] = round(mean_absolute_percentage_error(Y_test,Y_test_pred),4)
    

    model = sm.OLS(Y, X_const).fit()

    # Préparation du modèle en mémoire
    buffer = BytesIO()
    pickle.dump(model, buffer)
    buffer.seek(0)

    # Upload S3
    s3 = boto3.client("s3", region_name="eu-west-3")
    bucket_name = "geodechet"
    key = f"modeles/model_{target}.pkl"

    s3.upload_fileobj(buffer, bucket_name, key)


    return new_rows

# Chargement des données

In [4]:
s3 = boto3.client("s3", region_name="eu-west-3")

bucket_name = "geodechet"
key = "data/data_wip_complet.csv"  # chemin dans ton bucket

obj = s3.get_object(Bucket=bucket_name, Key=key)

df_origine = pd.read_csv(BytesIO(obj["Body"].read()), encoding="utf-8-sig")
df = df_origine.drop(['Code_Dpt','annee'], axis=1)
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Departement                                672 non-null    object 
 1   Region                                     672 non-null    object 
 2   densite                                    672 non-null    float64
 3   pop_globale                                672 non-null    int64  
 4   tranche_age_0-24                           672 non-null    int64  
 5   tranche_age_25-59                          672 non-null    int64  
 6   tranche_age_60+                            672 non-null    int64  
 7   csp1_agriculteurs                          672 non-null    int64  
 8   csp2_artisans_commercant_chef_entreprises  672 non-null    int64  
 9   csp3_cadres_professions_intellectuelles    672 non-null    int64  
 10  csp4_professions_intermedi

# Bloc principal

In [5]:
pd.options.mode.chained_assignment = None

df_results = pd.DataFrame(columns=[
    'cible', 
    'Best_alpha',
    'R2 train',
    'R2 test',
    'RMSE',
    'MAE',
    'MAPE', 
    'Valeur min',
    'Valeur max',
    'Moyenne',
    'Mediane'
])

targets = ['Deblais_gravats','Dechets_verts','Encombrants','Materiaux_recyclables']

nb_split = 4
nb_alphas = 100
random = 42
size = 0.1
iter = 50000


s3 = boto3.client("s3", region_name="eu-west-3")
bucket_name = "geodechet"

# 1️⃣ Filtre 2021
df_2021_envoi = df_origine[df_origine["annee"] == 2021].copy()

# 2️⃣ Upload data_wip.csv sur S3
csv_buffer = StringIO()
df_2021_envoi.to_csv(csv_buffer, index=False, encoding="utf-8-sig")

s3.put_object(
    Bucket=bucket_name,
    Key="data/data_wip.csv",
    Body=csv_buffer.getvalue(),
    ContentType="text/csv",
)

df_2021_envoi = df_2021_envoi.drop(['Code_Dpt', 'annee'], axis=1)

df_2021_dummies = pd.get_dummies(df_2021_envoi).astype(float)

csv_buffer2 = StringIO()
df_2021_dummies.to_csv(csv_buffer2, encoding="utf-8-sig", index=False)

s3.put_object(
    Bucket=bucket_name,
    Key="data/df_dummies.csv",
    Body=csv_buffer2.getvalue(),
    ContentType="text/csv",
)


for target in targets:
    print(f'{target} {datetime.datetime.now()}')
    new_rows = [
        {'cible': target, 
         'Valeur min':df[target].min(), 
         'Moyenne':df[target].mean().round(0), 
         'Valeur max':df[target].max(),
         'Mediane':df[target].median()
         }
    ]

    X,Y = select_target(target,df)
    
    X_encoded = pd.get_dummies(X).astype(float)

    scaler = StandardScaler()

    # Standardisation
    X_scaled = scaler.fit_transform(X_encoded)

    # Initialisation de KFold et des alphas
    kf = KFold(n_splits=nb_split, shuffle=True, random_state=random)
    alphas = np.logspace(-3, 2, nb_alphas)

    # Lancement du Lasso avec cross-validation
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        lasso = LassoCV(alphas=alphas, cv=kf, max_iter=iter,tol=1e-3)
        lasso.fit(X_scaled, Y)

    # Resultats
    best_alpha = lasso.alpha_
    coefficients = lasso.coef_
    intercept = lasso.intercept_

    print(f"✅ Meilleur alpha pour {target} : {best_alpha:.4f}")
        
    coefficients_series = pd.Series(coefficients, index=X_encoded.columns)
    selected_features = coefficients_series[coefficients_series != 0].index.tolist() 
    
    X_encoded = X_encoded[selected_features]

    new_rows = Select_features_PValue(X_encoded,Y,target,new_rows)

    new_row_df = pd.DataFrame(new_rows)

    # Pour forcer les colonnes à correspondre
    new_row_df = new_row_df.reindex(columns=df_results.columns)

    df_results = pd.concat([df_results, new_row_df], ignore_index=True)

    print(f'Generation du modèle et des colonnes pour {target} est finie, {datetime.datetime.now()}')

print('Fin de la generation des modèles')


Deblais_gravats 2025-11-14 18:26:57.794209
✅ Meilleur alpha pour Deblais_gravats : 7.7426


C:\Users\franc\AppData\Local\Temp\ipykernel_22328\3258273760.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_row_df], ignore_index=True)


Generation du modèle et des colonnes pour Deblais_gravats est finie, 2025-11-14 18:27:25.514253
Dechets_verts 2025-11-14 18:27:25.514306
✅ Meilleur alpha pour Dechets_verts : 2.4201
Generation du modèle et des colonnes pour Dechets_verts est finie, 2025-11-14 18:27:59.212077
Encombrants 2025-11-14 18:27:59.212181
✅ Meilleur alpha pour Encombrants : 4.8626
Generation du modèle et des colonnes pour Encombrants est finie, 2025-11-14 18:28:29.297680
Materiaux_recyclables 2025-11-14 18:28:29.297893
✅ Meilleur alpha pour Materiaux_recyclables : 0.0658
Generation du modèle et des colonnes pour Materiaux_recyclables est finie, 2025-11-14 18:28:54.265082
Fin de la generation des modèles


In [6]:
df_results[['cible','R2 train','R2 test','MAPE']]

,cible,R2 train,R2 test,MAPE
0,Deblais_gravats,0.971204,0.983183,0.1281
1,Dechets_verts,0.978239,0.967753,0.1290
2,Encombrants,0.958328,0.907644,0.1182
3,Materiaux_recyclables,0.981021,0.967526,0.1283
